
# AISE4010 Group 11 ? Stage 2 model workspace

Temporary workspace for developing RNN, LSTM, GRU, and TCN models. It keeps only the shared data/window preparation required to train these models; Stage 1 exploration, baseline/model runs, tables, and embedded data outputs are intentionally omitted.

Input contract: past 168 hourly steps -> next 24 hourly targets. Select candidates with validation-only, measured-hour midnight-origin RMSE. Copy the selected model, configuration, and validation evidence into the final evolving notebook, where Stage 1 models and the final test comparison belong.


In [1]:
import os

# Convert the raw UCI .txt (';'-separated, '?' = missing) to CSV once
if not os.path.exists('household_power_consumption.csv'):
    with open('household_power_consumption.txt', newline='') as src, open('household_power_consumption.csv', 'w', newline='') as dst:
        for line in src:
            dst.write(line.replace(';', ',').replace('?', ''))

In [2]:
import os, sys
os.environ['PATH'] = os.path.join(sys.prefix, 'Library', 'bin') + os.pathsep + os.environ.get('PATH', '')
import torch
print(sys.executable)
print(torch.backends.cudnn.version())

c:\Users\ahaze\anaconda3\envs\4010-DeepLearning\python.exe
92800


In [3]:

import pandas as pd

df = pd.read_csv('household_power_consumption.csv')
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
df = df.drop(columns=['Date', 'Time']).set_index('Datetime')
print('Loaded rows:', len(df), '| coverage:', df.index.min(), '->', df.index.max())


Loaded rows: 2075259 | coverage: 2006-12-16 17:24:00 -> 2010-11-26 21:02:00


### GPU check
Confirms which device PyTorch will train on. Every model below is moved to `device`.

In [4]:
import os
import random
import sys
import numpy as np

os.environ['PATH'] = os.path.join(sys.prefix, 'Library', 'bin') + os.pathsep + os.environ.get('PATH', '')
import torch

print('PyTorch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA version:', torch.version.cuda, '| cuDNN version:', torch.backends.cudnn.version())

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    print('Training on GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU found, training on CPU')

# A small matmul proves tensors actually land on the chosen device
probe = torch.randn(256, 256, device=device)
print('Test matmul ran on:', (probe @ probe).device)

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)  # also seeds CUDA

# Deterministic cuDNN kernels so GPU runs repeat exactly with the same seed
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
set_seed()

PyTorch 2.14.1
CUDA available: True
CUDA version: 13.4 | cuDNN version: 92800
Training on GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Test matmul ran on: cuda:0


In [5]:
TARGET = 'Global_active_power'
LOOKBACK, HORIZON = 168, 24
DAY, WEEK = 24, 168


## Resampling and imputation
* **Resample:** hourly mean. Hours with fewer than 30 recorded minutes are set to missing.
* **Short gaps (≤ 3 h):** forward fill, which carries the last observed hour forward.
* **Long gaps (up to about 5 days):** the value from the same hour one week earlier. Any remainder is forward-filled.

Every fill is **causal**: a missing hour is filled only from earlier hours. Interpolation would use the next observed hour, which could be a target hour of the same window. Nothing here is fitted to data either, so imputation cannot leak test statistics into training. The `imputed` mask is kept so we can report how much of each split is filled in.

In [6]:
counts = df[TARGET].resample('h').count()
hourly_raw = df.resample('h').mean()
hourly_raw.loc[counts < 30] = np.nan
imputed = hourly_raw[TARGET].isna()

# Length of each run of missing hours
run_id = (imputed != imputed.shift()).cumsum()
run_len = imputed.groupby(run_id).transform('size')
short_gap = imputed & (run_len <= 3)

hourly = hourly_raw.copy()
hourly.loc[short_gap] = hourly_raw.ffill().loc[short_gap]
for _ in range(2):  # a second pass covers gaps whose previous week was also missing
    hourly = hourly.fillna(hourly.shift(WEEK))
hourly = hourly.ffill().dropna()  # only leading hours with no history can remain; drop them
imputed = imputed.loc[hourly.index]

assert not hourly.isna().any().any()
print(f'Hourly rows: {len(hourly)} | imputed hours: {imputed.sum()} ({imputed.mean():.2%}), '
      f'of which short gaps: {short_gap.sum()}')

Hourly rows: 34589 | imputed hours: 432 (1.25%), of which short gaps: 7



## Shared forecast setup
This compact setup reproduces only the preprocessing and window definitions required by the Stage 2 models. The same chronological 60/20/20 split is used. Scalers and the profile baseline are fitted on training rows only. The chosen Stage 1 feature configuration is retained for a fair comparison when the selected Stage 2 model is later integrated into the final notebook.


In [7]:
features = hourly.copy()
idx = features.index
dow = idx.day_of_week.to_numpy()
year_phase = 2 * np.pi * ((idx.day_of_year.to_numpy() - 1) + idx.hour.to_numpy() / 24) / 365.2425
features['hour_sin'] = np.sin(2 * np.pi * idx.hour / 24)
features['hour_cos'] = np.cos(2 * np.pi * idx.hour / 24)
features['dow_sin'] = np.sin(2 * np.pi * dow / 7)
features['dow_cos'] = np.cos(2 * np.pi * dow / 7)
features['is_weekend'] = (dow >= 5).astype(float)
for harmonic in (1, 2):
    features[f'year_sin_{harmonic}'] = np.sin(harmonic * year_phase)
    features[f'year_cos_{harmonic}'] = np.cos(harmonic * year_phase)

past_gap = hourly[TARGET]
causal_mean = past_gap.shift(1).expanding(min_periods=1).mean().fillna(past_gap.iloc[0])
annual_index = idx - pd.DateOffset(years=1)
annual_lag = past_gap.reindex(annual_index).to_numpy()
features['GAP_lag_1y_available'] = np.isfinite(annual_lag).astype(float)
features['GAP_lag_1y'] = np.where(np.isfinite(annual_lag), annual_lag, causal_mean.to_numpy())
for days in (7, 28):
    trailing = past_gap.shift(1).rolling(days * 24, min_periods=24).mean()
    features[f'GAP_mean_{days}d'] = trailing.fillna(causal_mean)
FEATURES = list(features.columns)
TARGET_IDX = FEATURES.index(TARGET)

n = len(features)
train_end, val_end = int(n * 0.6), int(n * 0.8)
splits = {'train': (0, train_end), 'val': (train_end, val_end), 'test': (val_end, n)}

pd.DataFrame([{'split': s, 'start': features.index[a], 'end': features.index[b - 1], 'hours': b - a,
               'imputed %': round(imputed.iloc[a:b].mean() * 100, 2)}
              for s, (a, b) in splits.items()]).set_index('split')

,start,end,hours,imputed %
split,,,,
train,2006-12-16 17:00:00,2009-04-29 09:00:00,20753,0.33
val,2009-04-29 10:00:00,2010-02-11 15:00:00,6918,1.76
test,2010-02-11 16:00:00,2010-11-26 21:00:00,6918,3.50



### Causal seasonal baseline
The Stage 1-selected `profile` baseline (hour-of-week mean, fitted on training rows) is retained to match the established Stage 1 input/target representation. Calendar features and rolling/annual features are generated causally; no centered or future-looking features are used.


In [8]:

gap = hourly[TARGET].to_numpy()
how = (features.index.day_of_week * 24 + features.index.hour).to_numpy()
profile = pd.Series(gap[:train_end]).groupby(how[:train_end]).mean()
weekly_profile = profile.reindex(how).to_numpy()
train_residual = gap[:train_end] - weekly_profile[:train_end]
month_effect = pd.Series(train_residual).groupby(features.index.month[:train_end]).mean()
annual_adjustment = pd.Series(features.index.month).map(month_effect).fillna(0.0).to_numpy()
annual_profile = weekly_profile + annual_adjustment
seasonal_base = {
    'none': np.zeros(n),
    'week': np.r_[np.full(WEEK, np.nan), gap[:-WEEK]],
    'profile': weekly_profile,
    'profile_year': annual_profile,
}


In [9]:
from sklearn.preprocessing import StandardScaler

def to_kw(y, split):
    """Undo the target scaling and add the seasonal baseline back, so all metrics are in kW."""
    return y * x_scaler.scale_[TARGET_IDX] + x_scaler.mean_[TARGET_IDX] + base[split]

def make_windows(start, end):
    """Windows whose 24-h target lies in rows [start, end); the lookback may come from earlier rows."""
    # The 'week' residual needs one extra week of history; every variant uses the same windows so they compare fairly
    t = np.arange(max(start, LOOKBACK + WEEK), end - HORIZON + 1)
    X = np.stack([X_all[i - LOOKBACK:i] for i in t])
    y = np.stack([X_all[i:i + HORIZON, TARGET_IDX] for i in t])
    return X, y, t

def prepare(variant, drop):
    """Scale the inputs and build the windows for one seasonal baseline and column set (sets the globals used below)."""
    global FEATURES, TARGET_IDX, x_scaler, X_all, data, base, measured, daily_origin, Y_kw, X_train, y_train, X_val, y_val
    frame = features.drop(columns=drop)
    target_col = TARGET
    if variant != 'none':
        frame['GAP_resid'] = gap - seasonal_base[variant]  # raw GAP stays as a lagged input
        target_col = 'GAP_resid'
    FEATURES = list(frame.columns)
    TARGET_IDX = FEATURES.index(target_col)
    x_scaler = StandardScaler().fit(frame.iloc[:train_end])  # train rows only; NaNs (first week of 'week') are ignored
    X_all = x_scaler.transform(frame).astype(np.float32)

    data = {s: make_windows(a, b) for s, (a, b) in splits.items()}
    hours = {s: t[:, None] + np.arange(HORIZON) for s, (_, _, t) in data.items()}  # row index of every target hour
    base = {s: seasonal_base[variant][h] for s, h in hours.items()}
    measured = {s: ~imputed.to_numpy()[h] for s, h in hours.items()}
    daily_origin = {s: features.index[t].hour.to_numpy() == 0 for s, (_, _, t) in data.items()}
    Y_kw = {s: to_kw(y, s) for s, (_, y, _) in data.items()}

    X_train, y_train, t_train = data['train']
    X_val, y_val, t_val = data['val']
    assert X_train.shape[1:] == (LOOKBACK, len(FEATURES)) and y_train.shape[1] == HORIZON
    assert t_train[-1] + HORIZON <= train_end and t_val[-1] + HORIZON <= val_end  # no target crosses a split
    assert all(np.isfinite(X).all() for X, _, _ in data.values())
    for s, h in hours.items():  # adding the baseline back recovers the actual kW
        assert np.allclose(Y_kw[s], gap[h], atol=1e-3)

best_variant = 'profile'
best_drop = ['Global_intensity', 'Voltage', 'Global_reactive_power']
prepare(best_variant, best_drop)
print('Training windows:', X_train.shape, '| 24-hour targets:', y_train.shape[1])


Training windows: (20394, 168, 18) | 24-hour targets: 24


In [10]:
import copy
import time
from torch import nn

BATCH = 64
MAX_EPOCHS, PATIENCE = 100, 10

def evaluate(y_true, y_pred, mask=None):
    error = y_pred - y_true
    if mask is not None:
        error = error[mask]
    return {'MAE': np.abs(error).mean(), 'RMSE': np.sqrt(np.mean(error ** 2))}

def count_params(model):
    return sum(parameter.numel() for parameter in model.parameters())

def predict(model, X):
    model.eval()
    with torch.no_grad():
        batches = [model(batch.to(device)).cpu() for batch in torch.from_numpy(X).split(1024)]
    return torch.cat(batches).numpy()

def fit(build, train, val, lr=3e-4, batch_size=BATCH, weight_decay=1e-2,
        max_epochs=MAX_EPOCHS, patience=PATIENCE):
    """Train with validation-MSE early stopping and return the best checkpoint."""
    set_seed()
    model = build().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    X_train_device, y_train_device = (torch.from_numpy(array).to(device) for array in train)
    X_val_device, y_val_device = (torch.from_numpy(array).to(device) for array in val)
    history = {'train': [], 'val': []}
    best_loss, best_state, wait = float('inf'), copy.deepcopy(model.state_dict()), 0
    start = time.perf_counter()
    for _ in range(max_epochs):
        model.train()
        total_loss = 0.0
        for indices in torch.randperm(len(X_train_device), device=device).split(batch_size):
            optimizer.zero_grad()
            loss = loss_fn(model(X_train_device[indices]), y_train_device[indices])
            loss.backward()
            optimizer.step()
            total_loss += loss.detach() * len(indices)
        model.eval()
        with torch.no_grad():
            validation_loss = sum(
                loss_fn(model(x_batch), y_batch).item() * len(x_batch)
                for x_batch, y_batch in zip(X_val_device.split(1024), y_val_device.split(1024))
            ) / len(X_val_device)
        history['train'].append(total_loss.item() / len(X_train_device))
        history['val'].append(validation_loss)
        if validation_loss < best_loss:
            best_loss, wait = validation_loss, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    return model, history, time.perf_counter() - start


## Stage 2 ? recurrent models and TCN
Compare a vanilla RNN, LSTM, GRU, and TCN. All models use the same Stage 1-selected features, training/validation windows, MSE training loss, early stopping, and evaluation metrics. The HPO cell selects one Stage 2 architecture and configuration using validation RMSE only. This temporary notebook does not report test results; those belong in the final integrated notebook after model selection.


In [11]:
# Confirm the unchanged forecasting contract before fitting additional models.
assert X_train.shape[1] == LOOKBACK == 168
assert y_train.shape[1] == HORIZON == 24

class RecurrentForecaster(nn.Module):
    def __init__(self, cell='GRU', hidden=64, layers=1, dropout=0.1):
        super().__init__()
        recurrent = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}[cell]
        self.encoder = recurrent(
            input_size=len(FEATURES),
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            dropout=dropout if layers > 1 else 0.0,
        )
        self.head = nn.Linear(hidden, HORIZON)

    def forward(self, x):
        sequence, _ = self.encoder(x)
        return self.head(sequence[:, -1])

class CausalConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()
        self.left_padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(
            in_channels, out_channels, kernel_size,
            padding=self.left_padding, dilation=dilation,
        )

    def forward(self, x):
        output = self.conv(x)
        if self.left_padding:
            output = output[..., :-self.left_padding]
        return output

class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
        super().__init__()
        self.conv1 = CausalConv1d(in_channels, out_channels, kernel_size, dilation)
        self.conv2 = CausalConv1d(out_channels, out_channels, kernel_size, dilation)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.ReLU()
        self.skip = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        residual = self.skip(x)
        output = self.dropout(self.activation(self.conv1(x)))
        output = self.dropout(self.activation(self.conv2(output)))
        return self.activation(output + residual)

class TCNForecaster(nn.Module):
    def __init__(self, hidden=32, layers=6, dropout=0.1, kernel_size=3):
        super().__init__()
        blocks = []
        in_channels = len(FEATURES)
        for level in range(layers):
            blocks.append(TemporalBlock(
                in_channels, hidden, kernel_size, dilation=2 ** level, dropout=dropout,
            ))
            in_channels = hidden
        self.encoder = nn.Sequential(*blocks)
        self.head = nn.Linear(hidden, HORIZON)

    def forward(self, x):
        encoded = self.encoder(x.transpose(1, 2))
        return self.head(encoded[:, :, -1])

stage2_builders = {
    'RNN': lambda **config: RecurrentForecaster(cell='RNN', **config),
    'LSTM': lambda **config: RecurrentForecaster(cell='LSTM', **config),
    'GRU': lambda **config: RecurrentForecaster(cell='GRU', **config),
    'TCN': lambda **config: TCNForecaster(**config),
}

# The TCN's six kernel-3 blocks have a 253-hour receptive field, covering the full 168-hour input.
stage2_initial_configs = {
    'RNN': {'hidden': 64, 'layers': 1, 'dropout': 0.1},
    'LSTM': {'hidden': 64, 'layers': 1, 'dropout': 0.1},
    'GRU': {'hidden': 64, 'layers': 1, 'dropout': 0.1},
    'TCN': {'hidden': 32, 'layers': 6, 'dropout': 0.1, 'kernel_size': 3},
}


### Stage 2 model comparison
Each model consumes `(batch, 168, features)` and predicts `(batch, 24)`. The TCN uses causal dilated convolutions with a receptive field covering the one-week input. The table records validation MAE/RMSE, parameter count, training time, and inference time.


In [12]:
import os

stage2_names = list(stage2_builders)
stage2_models, stage2_times, stage2_rows = {}, {}, []
stage2_daily_mask = measured['val'] & daily_origin['val'][:, None]

for name in stage2_names:
    model, history, secs = fit(
        lambda name=name: stage2_builders[name](**stage2_initial_configs[name]),
        (X_train, y_train), (X_val, y_val),
        lr=3e-4, batch_size=BATCH, weight_decay=1e-2,
    )
    stage2_models[name] = model
    stage2_times[name] = secs
    prediction_kw = to_kw(predict(model, X_val), 'val')
    metrics = evaluate(Y_kw['val'], prediction_kw, stage2_daily_mask)
    stage2_rows.append({
        'model': name,
        'val MAE (daily starts)': metrics['MAE'],
        'val RMSE (daily starts)': metrics['RMSE'],
        'parameters': count_params(model),
        'training time (s)': secs,
        'epochs': len(history['val']),
    })

stage2_initial_results = pd.DataFrame(stage2_rows).set_index('model').sort_values('val RMSE (daily starts)')
def inference_ms_per_1000_windows(model, X, repeats=3):
    sample = X[:min(1024, len(X))]
    predict(model, sample)
    if device.type == 'cuda':
        torch.cuda.synchronize(device)
    start = time.perf_counter()
    for _ in range(repeats):
        predict(model, sample)
    if device.type == 'cuda':
        torch.cuda.synchronize(device)
    elapsed = time.perf_counter() - start
    return elapsed * 1_000_000 / (repeats * len(sample))

stage2_initial_results['inference (ms / 1000 windows)'] = [
    inference_ms_per_1000_windows(stage2_models[name], X_val)
    for name in stage2_initial_results.index
]
display(stage2_initial_results.round(4))

# Select the Stage 2 family to tune using validation RMSE only.
stage2_family = stage2_initial_results['val RMSE (daily starts)'].idxmin()
print('Stage 2 family selected for HPO:', stage2_family)


,val MAE (daily starts),val RMSE (daily starts),parameters,training time (s),epochs,inference (ms / 1000 windows)
model,,,,,,
TCN,0.4410,0.6092,37304,44.5414,14,16.0311
RNN,0.4506,0.6121,6936,25.2321,26,3.1271
GRU,0.4406,0.6140,17688,13.1986,15,6.1842
LSTM,0.4435,0.6161,23064,13.1597,14,5.2723


Stage 2 family selected for HPO: TCN



### Validation-only HPO
The initially best Stage 2 architecture is tuned with a fixed eight-trial budget. Width, depth, and learning rate vary; dropout, weight decay, batch size, seed, split, and windows are fixed. The selected checkpoint is saved separately from the Stage 1 checkpoint. Do not copy test metrics from this workspace; evaluate the locked choice in the final integrated notebook.


In [13]:
# Fixed, documented eight-trial validation-only search budget for the selected Stage 2 family.
if stage2_family == 'TCN':
    stage2_trials_config = [
        {'hidden': width, 'layers': depth, 'dropout': 0.1, 'kernel_size': 3, 'lr': lr}
        for width in (16, 32)
        for depth in (4, 6)
        for lr in (1e-3, 3e-4)
    ]
else:
    stage2_trials_config = [
        {'hidden': width, 'layers': depth, 'dropout': 0.1, 'lr': lr}
        for width in (32, 64)
        for depth in (1, 2)
        for lr in (1e-3, 3e-4)
    ]

stage2_trials, stage2_best = [], {'val RMSE': float('inf')}
for config in stage2_trials_config:
    model_config = {key: value for key, value in config.items() if key != 'lr'}
    model, history, secs = fit(
        lambda model_config=model_config: stage2_builders[stage2_family](**model_config),
        (X_train, y_train), (X_val, y_val),
        lr=config['lr'], batch_size=BATCH, weight_decay=1e-2,
    )
    prediction_kw = to_kw(predict(model, X_val), 'val')
    metrics = evaluate(Y_kw['val'], prediction_kw, stage2_daily_mask)
    trial = {
        **config,
        'val MAE (daily starts)': metrics['MAE'],
        'val RMSE (daily starts)': metrics['RMSE'],
        'epochs': len(history['val']),
        'training time (s)': secs,
    }
    stage2_trials.append(trial)
    if metrics['RMSE'] < stage2_best['val RMSE']:
        stage2_best = {
            'val RMSE': metrics['RMSE'],
            'val MAE': metrics['MAE'],
            'model': model,
            'config': model_config,
            'train_config': {'lr': config['lr'], 'batch_size': BATCH, 'weight_decay': 1e-2},
            'secs': secs,
        }

stage2_hpo_results = pd.DataFrame(stage2_trials).sort_values('val RMSE (daily starts)')
display(stage2_hpo_results.reset_index(drop=True).round(4))

stage2_tuned_name = f'{stage2_family} (Stage 2 tuned)'
stage2_tuned = stage2_best['model']
stage2_tuned_inference = inference_ms_per_1000_windows(stage2_tuned, X_val)

os.makedirs('models', exist_ok=True)
torch.save({
    'stage': 2,
    'family': stage2_family,
    'config': stage2_best['config'],
    'train_config': stage2_best['train_config'],
    'state_dict': stage2_tuned.state_dict(),
    'variant': best_variant,
    'dropped': best_drop,
    'lookback': LOOKBACK,
    'horizon': HORIZON,
    'target': TARGET,
    'features': FEATURES,
    'x_mean': x_scaler.mean_.tolist(),
    'x_scale': x_scaler.scale_.tolist(),
}, 'models/stage2_best.pt')

stage2_comparison = stage2_initial_results.copy()
stage2_comparison.loc[stage2_tuned_name] = {
    'val MAE (daily starts)': stage2_best['val MAE'],
    'val RMSE (daily starts)': stage2_best['val RMSE'],
    'parameters': count_params(stage2_tuned),
    'training time (s)': stage2_best['secs'],
    'epochs': np.nan,
    'inference (ms / 1000 windows)': stage2_tuned_inference,
}
print('Stage 2 validation comparison (measured hours, midnight-origin forecasts):')
display(stage2_comparison.round(4))
print(f"Saved validation-selected Stage 2 model ({stage2_family}) to models/stage2_best.pt")

# Reload the Stage 2 checkpoint and verify the 168-hour -> 24-hour forecast shape.
checkpoint = torch.load('models/stage2_best.pt', map_location=device)
reloaded_stage2 = stage2_builders[checkpoint['family']](**checkpoint['config']).to(device)
reloaded_stage2.load_state_dict(checkpoint['state_dict'])
smoke_forecast = predict(reloaded_stage2, X_val[:8])
assert smoke_forecast.shape == (8, HORIZON) and np.isfinite(smoke_forecast).all()
print('Stage 2 checkpoint smoke test passed:', smoke_forecast.shape)


,hidden,layers,dropout,kernel_size,lr,val MAE (daily starts),val RMSE (daily starts),epochs,training time (s)
0,16,6,0.1,3,0.0003,0.4354,0.6059,20,57.6977
1,32,4,0.1,3,0.0003,0.4367,0.6083,16,29.9250
2,16,6,0.1,3,0.0010,0.4356,0.6086,14,42.2822
3,32,6,0.1,3,0.0003,0.4410,0.6092,14,34.9256
4,32,4,0.1,3,0.0010,0.4381,0.6101,12,21.9287
5,32,6,0.1,3,0.0010,0.4370,0.6101,12,29.9275
6,16,4,0.1,3,0.0003,0.4395,0.6118,25,54.5746
7,16,4,0.1,3,0.0010,0.4431,0.6131,15,32.9527


Stage 2 validation comparison (measured hours, midnight-origin forecasts):


,val MAE (daily starts),val RMSE (daily starts),parameters,training time (s),epochs,inference (ms / 1000 windows)
model,,,,,,
TCN,0.4410,0.6092,37304,44.5414,14.0,16.0311
RNN,0.4506,0.6121,6936,25.2321,26.0,3.1271
GRU,0.4406,0.6140,17688,13.1986,15.0,6.1842
LSTM,0.4435,0.6161,23064,13.1597,14.0,5.2723
TCN (Stage 2 tuned),0.4354,0.6059,10216,57.6977,NaN,6.1803


Saved validation-selected Stage 2 model (TCN) to models/stage2_best.pt
Stage 2 checkpoint smoke test passed: (8, 24)
